# Three-Model Ensemble V2 — XGBoost Training & Hyperparameter Search

## Contents

| Section | Description |
|---------|-------------|
| 1. Imports & Paths | Environment setup and artifact paths |
| 2. Data Load & Feature Engineering | Load raw data, compute derived features |
| 3. Feature Set Definitions | Feature lists for each sub-model |
| 4. Quarterly Rolling 2-Year Folds | Walk-forward fold construction (504-day train, 63-day test, 5-day gap) |
| 5. Training Helpers | Fit and evaluation utilities |
| 6. Hyperparameter Grid Search | Grid search on folds 4–10; best params saved per model |
| 7. Save Artifacts | Write `best_params_*.json` for use by the performance notebook |

**Feature set (V2 final)**

| Model | Count | Features |
|-------|-------|----------|
| Price | 6 | `td_return_5d`, `td_return_20d`, `td_vs_xfn_5d`, `td_vs_xfn_20d`, `td_dist_52w_high_v2`, `td_volatility_20d` |
| Transcript | 10 | Exec tone, analyst Q&A sentiment, framing gap, topic shares/sentiments (guidance, AML, credit quality), `days_since_call` |
| News | 5 | `news_sent_mean_7d/30d`, `news_count_7d/30d`, `days_since_last_news` |


## 1. Imports & Paths

In [ ]:
import sys
import warnings
import pickle
import json
from pathlib import Path
from datetime import date
from itertools import product as iproduct

import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import f1_score
from IPython.display import display

warnings.filterwarnings('ignore')

NOTEBOOK_DIR    = Path('').resolve()          # final_model/notebooks/ when run via nbconvert or Jupyter
FINAL_MODEL_DIR = NOTEBOOK_DIR.parent        # final_model/
ROOT            = FINAL_MODEL_DIR.parents[1] # project root (step3_predictive_model -> project root)
EXP_PARENT      = ROOT / 'step3_predictive_model/model_experiments_archive'
STEP3_DIR       = ROOT / 'step3_predictive_model'
for p in [str(ROOT), str(EXP_PARENT), str(STEP3_DIR)]:
    if p not in sys.path:
        sys.path.insert(0, p)

ARTIFACT_DIR = FINAL_MODEL_DIR / 'artifacts'
ARTIFACT_DIR.mkdir(exist_ok=True)
print(f'ROOT          : {ROOT}')
print(f'ARTIFACT_DIR  : {ARTIFACT_DIR}')

## 2. Data Load & Feature Engineering

New vs V1:
- Load TD and XFN raw prices to compute 60d returns, 60d volume change, 52w-high distance (v2 formula)
- Compute `td_vs_xfn_20d` inline from two parquet columns
- Parse `news.jsonl` to build 90d rolling sentiment and count

In [ ]:
from redesign_single_stock.src.run_redesign_experiments import load_dataset, label_3class
from src.models.walk_forward_config import STRIDE_EVAL  # step3_predictive_model/src/models/

df = load_dataset()

# ── td_dist_52w_high_v2 from TD raw prices ───────────────────────────────
td_raw = pd.read_parquet(ROOT / 'step1_data_collection/data/raw/prices/TD_TO.parquet')
td_raw['date'] = pd.to_datetime(td_raw['date'])
td_raw = td_raw.sort_values('date').reset_index(drop=True)
high_252 = td_raw['adj_close'].rolling(252).max()
td_raw['td_dist_52w_high_v2'] = (td_raw['adj_close'] - high_252) / td_raw['adj_close']
df = df.merge(td_raw[['date', 'td_dist_52w_high_v2']], on='date', how='left')

# ── td_vs_xfn_20d: both columns already in parquet ───────────────────────
df['td_vs_xfn_20d'] = df['td_return_20d'] - df['xfn_return_20d']

# ── Model 2 — undecayed NLP features (CEO+CFO combined, no call_decay) ───
df['exec_tone_ffill'] = df[['transcript_ceo_prep_sentiment_mean_ffill',
                             'transcript_cfo_prep_sentiment_mean_ffill']].mean(axis=1)

print(f'Loaded {len(df)} rows  |  {df["date"].min().date()} to {df["date"].max().date()}')
print(f'Engineered: td_dist_52w_high_v2, td_vs_xfn_20d, exec_tone_ffill')


## 3. Feature Set Definitions

In [ ]:
TARGET    = 'target_excess_xfn_5d'
THRESHOLD = 0.003
LABEL_MAP = {-1: 0, 0: 1, 1: 2}
INV_MAP   = {0: -1, 1: 0, 2: 1}
STRIDE    = STRIDE_EVAL   # 5

TRAIN_WINDOW_DAYS = 504
QUARTER_DAYS      = 63
GAP_DAYS          = 5

# ── Model 1 — Price V2c (6 features) ────────────────────────────────────
# 5d/20d returns + sector alpha + 52w distance + volatility regime signal.
# Dropped vs V1: volume features. Dropped vs V2 7-feat: td_return_60d, td_vs_xfn_60d.
# Added back vs V2 5-feat: td_volatility_20d (realised vol; regime signal for ±0.3% threshold).
FEATURES_PRICE = [
    'td_return_5d',
    'td_return_20d',
    'td_vs_xfn_5d',
    'td_vs_xfn_20d',
    'td_dist_52w_high_v2',
    'td_volatility_20d',
]

# ── Model 2 — Transcript + Report (10 features, unchanged) ───────────────
FEATURES_TRANSCRIPT = [
    'exec_tone_ffill',
    'transcript_analyst_qa_sentiment_mean_ffill',
    'framing_gap_ffill',
    'topic_guidance_share_ffill',
    'topic_guidance_sentiment_ffill',
    'topic_regulatory_AML_share_ffill',
    'topic_regulatory_AML_sentiment_ffill',
    'topic_credit_quality_share_ffill',
    'topic_credit_quality_sentiment_ffill',
    'days_since_call',
]

# ── Model 3 — News (5 features) ──────────────────────────────────────────
FEATURES_NEWS = [
    'news_sent_mean_7d',
    'news_sent_mean_30d',
    'news_count_7d',
    'news_count_30d',
    'days_since_last_news',
]

ALL_FEATURES = {
    'price':      FEATURES_PRICE,
    'transcript': FEATURES_TRANSCRIPT,
    'news':       FEATURES_NEWS,
}

for model_name, feats in ALL_FEATURES.items():
    missing = [f for f in feats if f not in df.columns]
    assert not missing, f'[{model_name}] Missing columns: {missing}'
    print(f'[{model_name}]  {len(feats)} features — OK')

print(f'\nTarget column present: {TARGET in df.columns}')
print(f'STRIDE = {STRIDE}')


## 4. Quarterly Rolling 2-Year Folds (5-Day Gap)

In [ ]:
def make_quarterly_rolling2y_folds(df, target,
                                   train_days=TRAIN_WINDOW_DAYS,
                                   quarter_days=QUARTER_DAYS,
                                   gap=GAP_DAYS):
    dated = df[df[target].notna()].sort_values('date')
    dates = dated['date'].values
    folds = []
    i, fold_id = train_days + gap, 1
    while i < len(dates):
        te_end_i      = min(i + quarter_days - 1, len(dates) - 1)
        train_end_idx = i - 1 - gap
        tr_s_i        = max(0, train_end_idx + 1 - train_days)
        folds.append((
            fold_id,
            str(dates[tr_s_i])[:10],
            str(dates[train_end_idx])[:10],
            str(dates[i])[:10],
            str(dates[te_end_i])[:10],
        ))
        i = te_end_i + 1
        fold_id += 1
    return folds

FOLDS = make_quarterly_rolling2y_folds(df, TARGET)

print(f'Total quarterly folds: {len(FOLDS)}')
print(f'{"Fold":<6} {"Train start":<13} {"Train end":<13} {"Test start":<13} {"Test end":<13}'
      f' {"Gap":>5} {"Train rows":>11} {"Test rows":>9}')
print('-' * 82)
for fold_id, tr_s, tr_e, te_s, te_e in FOLDS:
    tr = df[(df['date'] >= pd.Timestamp(tr_s)) &
            (df['date'] <= pd.Timestamp(tr_e)) & df[TARGET].notna()]
    te = df[(df['date'] >= pd.Timestamp(te_s)) &
            (df['date'] <= pd.Timestamp(te_e)) & df[TARGET].notna()]
    gap_cal = (pd.Timestamp(te_s) - pd.Timestamp(tr_e)).days
    print(f'{fold_id:<6} {tr_s:<13} {tr_e:<13} {te_s:<13} {te_e:<13}'
          f' {gap_cal:>5} {len(tr):>11} {len(te):>9}')

## 5. Training Helpers

In [ ]:
def fit_xgb(train_df, test_df, features, params):
    """Fit XGBoost, return (clf, hard_preds, y_true_cls, y_true_cont, prob_matrix)."""
    x_tr   = train_df[features].fillna(train_df[features].median())
    y_tr   = label_3class(train_df[TARGET].values, threshold=THRESHOLD)
    y_enc  = np.array([LABEL_MAP[v] for v in y_tr], dtype=int)

    clf = xgb.XGBClassifier(
        objective='multi:softprob', num_class=3,
        random_state=42, n_jobs=1, verbosity=0, **params
    )
    clf.fit(x_tr, y_enc)

    x_te       = test_df[features].fillna(train_df[features].median())
    y_te_cls   = label_3class(test_df[TARGET].values, threshold=THRESHOLD)
    y_te_cont  = test_df[TARGET].values
    prob_matrix = clf.predict_proba(x_te)
    hard_preds  = np.array([INV_MAP[int(p)] for p in clf.predict(x_te)])

    return clf, hard_preds, y_te_cls, y_te_cont, prob_matrix


def offset_metrics(preds, y_true_cls, y_true_cont):
    """Stride-offset averaging to de-bias evaluation from fold-boundary effects."""
    rows = []
    for off in range(STRIDE):
        idx = np.arange(off, len(preds), STRIDE)
        p, y_cls, y_cont = preds[idx], y_true_cls[idx], y_true_cont[idx]
        active = (p != 0)
        rows.append({
            'mean_acc':   (p == y_cls).mean(),
            'macro_f1':   f1_score(y_cls, p, average='macro',
                                   zero_division=0, labels=[-1, 0, 1]),
            'active_cov': active.mean(),
            'active_asa': float((np.sign(p[active]) == np.sign(y_cont[active])).mean())
                          if active.sum() > 0 else np.nan,
        })
    return pd.DataFrame(rows).mean()


print('Helpers defined.')

## 6. Hyperparameter Grid Search

Grid is designed for ~504 training rows and 7–10 features.  
Shallower trees and higher `min_child_weight` vs the reference 15-feature model to control overfitting.  
L1 (`reg_alpha`) and L2 (`reg_lambda`) regularisation are searched explicitly.

**Grid**: 3 × 3 × 3 × 2 × 2 = **108 configs per model**, evaluated on folds 4–10 (7 middle folds).  
**Fixed**: `learning_rate=0.05`, `subsample=0.8`, `colsample_bytree=0.8`, `tree_method='hist'`

In [ ]:
GRID = {
    'max_depth':        [2, 3, 4],
    'min_child_weight': [10, 20, 30],
    'n_estimators':     [50, 80, 120],
    'reg_alpha':        [0.0, 0.2],
    'reg_lambda':       [1.0, 2.0],
}
FIXED_PARAMS = dict(
    learning_rate    = 0.05,
    subsample        = 0.8,
    colsample_bytree = 0.8,
    tree_method      = 'hist',
)

n_folds    = len(FOLDS)
tune_start = max(2, n_folds // 4)
tune_end   = min(n_folds - 2, n_folds * 3 // 4)
TUNE_FOLDS = FOLDS[tune_start : tune_end + 1]

param_combos = list(iproduct(
    GRID['max_depth'], GRID['min_child_weight'], GRID['n_estimators'],
    GRID['reg_alpha'], GRID['reg_lambda']
))
print(f'Tuning on folds {tune_start+1}..{tune_end+1} ({len(TUNE_FOLDS)} of {n_folds} total)')
print(f'Grid: {len(param_combos)} configs × {len(TUNE_FOLDS)} tuning folds × 3 models')

BEST_PARAMS = {}

for model_name, features in ALL_FEATURES.items():
    print(f'\n── Grid search: {model_name} ({len(features)} features) ──')
    search_rows = []

    for md, mcw, ne, alpha, lam in param_combos:
        params = dict(
            max_depth=md, min_child_weight=mcw, n_estimators=ne,
            reg_alpha=alpha, reg_lambda=lam,
            **FIXED_PARAMS
        )
        fold_metrics = []
        for fold_id, tr_s, tr_e, te_s, te_e in TUNE_FOLDS:
            train_df = df[(df['date'] >= pd.Timestamp(tr_s)) &
                          (df['date'] <= pd.Timestamp(tr_e)) & df[TARGET].notna()]
            test_df  = df[(df['date'] >= pd.Timestamp(te_s)) &
                          (df['date'] <= pd.Timestamp(te_e)) & df[TARGET].notna()]
            if len(test_df) < 10:
                continue
            _, preds, y_cls, y_cont, _ = fit_xgb(train_df, test_df, features, params)
            fold_metrics.append(offset_metrics(preds, y_cls, y_cont))

        if not fold_metrics:
            continue
        agg = pd.DataFrame(fold_metrics).mean()
        search_rows.append({
            'max_depth': md, 'min_child_weight': mcw, 'n_estimators': ne,
            'reg_alpha': alpha, 'reg_lambda': lam,
            'active_asa': agg['active_asa'], 'macro_f1': agg['macro_f1'],
            'active_cov': agg['active_cov'],
            'composite': 0.6 * agg['active_asa'] + 0.4 * agg['macro_f1'],
        })

    search_df = pd.DataFrame(search_rows).sort_values('composite', ascending=False)
    best_row  = search_df.iloc[0]
    best      = dict(
        max_depth        = int(best_row['max_depth']),
        min_child_weight = int(best_row['min_child_weight']),
        n_estimators     = int(best_row['n_estimators']),
        reg_alpha        = float(best_row['reg_alpha']),
        reg_lambda       = float(best_row['reg_lambda']),
        **FIXED_PARAMS
    )
    BEST_PARAMS[model_name] = best

    print('Top-5 configs:')
    display(search_df.head(5).round(4))
    print(f'Best params : {best}')
    print(f'Composite   : {best_row["composite"]:.4f}')

## 7. Save Artifacts

In [ ]:
today = date.today().isoformat()

params_path = ARTIFACT_DIR / f'best_params_{today}.json'
save_obj = {
    'date': today,
    'version': 'v2',
    'target': TARGET,
    'threshold': THRESHOLD,
    'train_window_days': TRAIN_WINDOW_DAYS,
    'quarter_days': QUARTER_DAYS,
    'gap_days': GAP_DAYS,
    'stride': STRIDE,
    'features': ALL_FEATURES,
    'best_params': BEST_PARAMS,
}
with open(params_path, 'w') as f:
    json.dump(save_obj, f, indent=2)
print(f'Best params saved → {params_path}')

for model_name, features in ALL_FEATURES.items():
    full_df  = df[df[TARGET].notna()]
    x_full   = full_df[features].fillna(full_df[features].median())
    y_full   = label_3class(full_df[TARGET].values, threshold=THRESHOLD)
    y_enc    = np.array([LABEL_MAP[v] for v in y_full], dtype=int)

    clf_full = xgb.XGBClassifier(
        objective='multi:softprob', num_class=3,
        random_state=42, n_jobs=1, verbosity=0,
        **BEST_PARAMS[model_name]
    )
    clf_full.fit(x_full, y_enc)

    pkl_path = ARTIFACT_DIR / f'xgb_ensemble_v2_{model_name}_{today}.pkl'
    with open(pkl_path, 'wb') as f:
        pickle.dump({'model': clf_full, 'features': features,
                     'params': BEST_PARAMS[model_name]}, f)
    print(f'[{model_name}] Full-data model saved → {pkl_path}')

print('\nAll artifacts saved.')